In [ ]:
import torch
import numpy as np
from sbi_particle_physics.objects.model import Model
from sbi_particle_physics.objects.normalizer import Normalizer
from sbi_particle_physics.managers.plotter import Plotter
from sbi_particle_physics.managers.backup import Backup
from sbi_particle_physics.config import DATA_DIR, ACCEPTANCE_COEFFS_PATH, REAL_DATA
from sbi_particle_physics.managers.real_data import RealData
from sbi_particle_physics.managers.imperfections_diagnostics import ImperfectionsDiagnostics
from sbi_particle_physics.objects.imperfections import Imperfections

In [ ]:
device = "cpu" # this small test works on cpu
n_points = 5
n_samples = 5
model = Model(device, n_points)

model.set_prior_basic([3], [5])
model.set_simulator(stride=2, pre_N=2, preruns=2, use_imperfections=True)

raw_data, raw_parameters = model.simulate_raw_data(n_samples=n_samples, n_points=n_points)
model.set_normalizer_with_data(raw_data=raw_data)
model.build_default()
data = model.normalizer.normalize_data(raw_data)
parameters = model.normalizer.normalize_parameters(raw_parameters)
model.append_data(data, parameters)
model.train(max_num_epochs=2, stop_after_epochs=1)
print("All done")

print("normalized stats", data.mean(dim=(0,1)), data.std(dim=(0,1)))

In [ ]:
print("Raw data")
print(raw_data.shape)
print(raw_data[0,:10])
unique_points, counts = torch.unique(
    raw_data[0],
    dim=0,
    return_counts=True
)
print("#unique points", unique_points.shape[0])

print("\nRaw parameters")
print(raw_parameters.shape)
print(raw_parameters[:10])

In [ ]:
Plotter.plot_a_sample(data[0], parameters[0])

In [ ]:
# Check whats the shape of the acceptance function
N = 10000000
u = torch.rand(N, 5, device=device)
u[:, 0] = 1.1 + (6.0 - 1.1) * u[:, 0] # [1.1, 6]
u[:, 1] = -1.0 + 2.0 * u[:, 1] # [-1, 1]
u[:, 2] = -1.0 + 2.0 * u[:, 2] # [-1, 1]
u[:, 3] = -np.pi + 2 * np.pi * u[:, 3] # [-π, π]
u[:, 4] = 5.0 + 0.1 * u[:, 4] # [5.0, 5.1]

v = model.simulator.imperfections._apply_acceptance(u)
print(f"v shape {v.shape}")

In [ ]:
ImperfectionsDiagnostics._plot_observables_2({"Acceptance": v})

In [ ]:
datasets = ImperfectionsDiagnostics.get_datasets(model, n_points=50)

In [ ]:
ImperfectionsDiagnostics.compare_datasets(datasets)


In [ ]:
ImperfectionsDiagnostics.chi2_test(datasets["Full"], datasets["Ideal"], bins=40)

In [ ]:
background = model.simulator.imperfections._sample_background_events(n=1000000)

In [ ]:
ImperfectionsDiagnostics._plot_observables_2({"Background": background})

In [ ]:
Plotter.plot_a_sample(background, torch.as_tensor([0], device=device))